In [ ]:
# ==============================================================================
# Cell 1: Environment Setup, Imports, Seeds, and Device Configuration
# ==============================================================================
import os
import sys
import copy
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

import iml_assignment as iml
from iml_assignment.config import (
    CNN_BATCH_SIZE,
    CNN_EPOCHS,
    CNN_LEARNING_RATE,
    CNN_WEIGHT_DECAY,
    CNN_NUM_FOLDS,
    CNN_SEED,
    CNN_KERNEL_SIZE,
    CNN_CONV_CHANNELS,
    CNN_DENSE_UNITS,
    CNN_DROPOUT,
    CNN_DROPOUT_HEAD,
    CNN_GRAD_CLIP,
    CNN_ETA_MIN,
    CNN_CONFIG,
    LABEL2ID,
    ID2LABEL,
    ACTIVITY_CLASSES,
    NUM_CHANNELS,
    WINDOW_SIZE,
)

# Set deterministic random seeds across Python, NumPy, and PyTorch for full reproducibility
def set_seed(seed=CNN_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CNN_SEED)

# Detect computation device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Display active 1D-CNN hyperparameters loaded from src/iml_assignment/config.py
print("\n--- Active 1D-CNN Hyperparameters (Configurable in src/iml_assignment/config.py) ---")
for k, v in CNN_CONFIG.items():
    print(f"  {k:15s}: {v}")


In [ ]:
# ==============================================================================
# Cell 2: Data Loading, Spline Imputation, Reshaping (N, 23, 250), & Z-Score Normalization
# ==============================================================================
# 1. Load raw datasets using project library
train_df = iml.load_data("train")
test_df = iml.load_data("test")
print(f"Loaded Raw Data -> Train: {train_df.shape}, Test: {test_df.shape}")

# 2. Impute sensor dropouts (sentinel 9999.0) using cubic spline interpolation
clean_train_df = iml.clean_dataset(train_df, method="spline")
clean_test_df = iml.clean_dataset(test_df, method="spline")
print("Cubic spline imputation complete (0 sentinels / 0 NaNs remaining).")

# 3. Extract sensor feature columns and target labels using library
sensor_cols = iml.get_sensor_columns(clean_train_df)
y_train_labels = clean_train_df["target"].values
y_train = clean_train_df["target"].map(LABEL2ID).values.astype(np.int64)

X_train_flat = clean_train_df[sensor_cols].values.astype(np.float32)
X_test_flat = clean_test_df[sensor_cols].values.astype(np.float32)

# 4. Reshape flat (N, 5750) -> (N, 250 timesteps, 23 channels) -> Transpose to (N, 23 channels, 250 timesteps)
# PyTorch Conv1d expects shape: (Batch_Size, Channels, Length)
X_train = X_train_flat.reshape(-1, WINDOW_SIZE, NUM_CHANNELS).transpose(0, 2, 1)
X_test = X_test_flat.reshape(-1, WINDOW_SIZE, NUM_CHANNELS).transpose(0, 2, 1)

# 5. Channel-wise Z-score Normalization:
# Compute mean and standard deviation per channel across batch and time dimensions (axes 0 and 2)
# Strictly fit statistics on training data only to avoid test-set data leakage
ch_mean = X_train.mean(axis=(0, 2), keepdims=True)
ch_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-8

X_train_norm = (X_train - ch_mean) / ch_std
X_test_norm = (X_test - ch_mean) / ch_std

print(f"\nFinal Preprocessed Tensor Shapes:")
print(f"  X_train: {X_train_norm.shape}  | dtype: {X_train_norm.dtype}")
print(f"  y_train: {y_train.shape}       | classes: {len(np.unique(y_train))}")
print(f"  X_test:  {X_test_norm.shape}   | dtype: {X_test_norm.dtype}")


In [ ]:
# ==============================================================================
# Cell 3: PyTorch Dataset Definition
# ==============================================================================
class HARDataset(Dataset):
    """
    PyTorch Dataset wrapper for multi-channel wearable sensor time-series windows.
    Returns:
      - X: FloatTensor of shape (Channels=23, Timesteps=250)
      - y: LongTensor scalar containing target activity class ID (0 to 11), or None for test data.
    """
    def __init__(self, X: np.ndarray, y: np.ndarray = None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# Quick sanity check
demo_dataset = HARDataset(X_train_norm[:4], y_train[:4])
demo_x, demo_y = demo_dataset[0]
print(f"HARDataset Sanity Check: Sample 0 shape = {demo_x.shape}, Label = {demo_y.item()} ({ID2LABEL[demo_y.item()]})")


In [ ]:
# ==============================================================================
# Cell 4: 1D-CNN Architecture Definition (Parametric from config.py)
# ==============================================================================
class ConvBlock1D(nn.Module):
    """
    Dual Conv1D block with Batch Normalization, ReLU activation, and Max Pooling.
    Flow: Conv1D -> BatchNorm -> ReLU -> Conv1D -> BatchNorm -> ReLU -> MaxPool1D
    """
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = CNN_KERNEL_SIZE):
        super().__init__()
        padding = kernel_size // 2  # Same padding to preserve temporal length before pooling
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=padding, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, padding=padding, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class HAR1DCNN(nn.Module):
    """
    Parametric 1D Convolutional Neural Network for Multimodal Wearable Sensor HAR.
    
    Architecture:
      - Sequential ConvBlock1D stages (channels: 23 -> 64 -> 128 -> 256)
      - Global Adaptive Average Pooling (collapses temporal dimension from L to 1)
      - Classification Head: Dropout -> Dense(128) -> BatchNorm -> ReLU -> Dropout -> Linear(12)
    """
    def __init__(
        self,
        in_channels: int = NUM_CHANNELS,
        num_classes: int = len(ACTIVITY_CLASSES),
        conv_channels: tuple = CNN_CONV_CHANNELS,
        kernel_size: int = CNN_KERNEL_SIZE,
        dense_units: int = CNN_DENSE_UNITS,
        dropout: float = CNN_DROPOUT,
        dropout_head: float = CNN_DROPOUT_HEAD,
    ):
        super().__init__()
        
        # Build convolutional backbone dynamically based on configured channel list
        blocks = []
        c_in = in_channels
        for c_out in conv_channels:
            blocks.append(ConvBlock1D(c_in, c_out, kernel_size=kernel_size))
            c_in = c_out
        self.backbone = nn.Sequential(*blocks)
        
        # Global pooling collapses the temporal dimension: (B, C, L) -> (B, C, 1) -> (B, C)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Multilayer perceptron classification head
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(conv_channels[-1], dense_units, bias=False),
            nn.BatchNorm1d(dense_units),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_head),
            nn.Linear(dense_units, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (Batch_Size, 23 channels, 250 timesteps)
        feats = self.backbone(x)
        pooled = self.global_pool(feats).flatten(1)  # (Batch_Size, conv_channels[-1])
        logits = self.head(pooled)                   # (Batch_Size, num_classes)
        return logits

# Instantiate model, inspect parameter count, and perform dummy forward pass
model_test = HAR1DCNN().to(device)
total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
dummy_input = torch.randn(2, NUM_CHANNELS, WINDOW_SIZE, device=device)
dummy_output = model_test(dummy_input)

print(f"HAR1DCNN Architecture Initialized:")
print(f"  Input Shape:  {tuple(dummy_input.shape)}")
print(f"  Output Shape: {tuple(dummy_output.shape)} (Logits for 12 classes)")
print(f"  Trainable Parameters: {total_params:,}")


In [ ]:
# ==============================================================================
# Cell 5: Training & Validation Epoch Functions
# ==============================================================================
def train_one_epoch(model, dataloader, criterion, optimizer, scheduler, device, grad_clip=CNN_GRAD_CLIP):
    """Executes a single training epoch with backpropagation, gradient clipping, and scheduler step."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        # Gradient clipping prevents gradient explosion during early conv training
        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        optimizer.step()
        running_loss += loss.item() * inputs.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    if scheduler is not None:
        scheduler.step()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate_model(model, dataloader, criterion, device):
    """Evaluates model performance on validation data without tracking gradients."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    all_probs = []

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        running_loss += loss.item() * inputs.size(0)

        probs = F.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    total = len(dataloader.dataset)
    epoch_loss = running_loss / total
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probs, axis=0)
    epoch_acc = (y_pred == y_true).mean()

    return epoch_loss, epoch_acc, y_pred, y_prob


In [ ]:
# ==============================================================================
# Cell 6: Stratified 5-Fold Cross-Validation Training Loop
# ==============================================================================
set_seed(CNN_SEED)

# Compute balanced class weights to handle slight activity class imbalances
class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class Weights Computed: Min={class_weights.min():.3f}, Max={class_weights.max():.3f}")

# Cross-validation splitter
skf = StratifiedKFold(n_splits=CNN_NUM_FOLDS, shuffle=True, random_state=CNN_SEED)

oof_preds = np.zeros(len(y_train), dtype=np.int64)
oof_probs = np.zeros((len(y_train), len(ACTIVITY_CLASSES)), dtype=np.float32)
test_ensemble_probs = np.zeros((len(X_test_norm), len(ACTIVITY_CLASSES)), dtype=np.float32)

test_dataset = HARDataset(X_test_norm)
test_loader = DataLoader(test_dataset, batch_size=CNN_BATCH_SIZE, shuffle=False)

cv_histories = []
start_time = time.time()

print(f"\nStarting {CNN_NUM_FOLDS}-Fold Cross Validation ({CNN_EPOCHS} epochs per fold)...\n")
print(f"{'Fold':<6} | {'Train Loss':<10} | {'Train Acc':<10} | {'Val Loss':<10} | {'Val Acc':<10} | {'Time':<6}")
print("-" * 60)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_norm, y_train)):
    fold_start = time.time()
    
    # 1. Prepare Datasets & DataLoaders for this fold
    train_fold_ds = HARDataset(X_train_norm[train_idx], y_train[train_idx])
    val_fold_ds = HARDataset(X_train_norm[val_idx], y_train[val_idx])

    train_loader = DataLoader(train_fold_ds, batch_size=CNN_BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_fold_ds, batch_size=CNN_BATCH_SIZE, shuffle=False)

    # 2. Instantiate fresh model, optimizer, scheduler, and criterion
    model = HAR1DCNN(
        in_channels=NUM_CHANNELS,
        num_classes=len(ACTIVITY_CLASSES),
        conv_channels=CNN_CONV_CHANNELS,
        kernel_size=CNN_KERNEL_SIZE,
        dense_units=CNN_DENSE_UNITS,
        dropout=CNN_DROPOUT,
        dropout_head=CNN_DROPOUT_HEAD,
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CNN_LEARNING_RATE, weight_decay=CNN_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CNN_EPOCHS, eta_min=CNN_ETA_MIN)

    # 3. Epoch Loop with Best Validation Tracking
    best_val_acc = 0.0
    best_val_loss = float("inf")
    best_model_weights = None
    fold_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, CNN_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device, CNN_GRAD_CLIP)
        va_loss, va_acc, _, _ = evaluate_model(model, val_loader, criterion, device)

        fold_history["train_loss"].append(tr_loss)
        fold_history["train_acc"].append(tr_acc)
        fold_history["val_loss"].append(va_loss)
        fold_history["val_acc"].append(va_acc)

        # Save weights on improvement (prioritize validation accuracy, break ties with validation loss)
        if va_acc > best_val_acc or (va_acc == best_val_acc and va_loss < best_val_loss):
            best_val_acc = va_acc
            best_val_loss = va_loss
            best_model_weights = copy.deepcopy(model.state_dict())

    # 4. Load best weights and compute final OOF predictions for this fold
    model.load_state_dict(best_model_weights)
    _, fold_acc, fold_pred, fold_prob = evaluate_model(model, val_loader, criterion, device)
    oof_preds[val_idx] = fold_pred
    oof_probs[val_idx] = fold_prob

    # 5. Predict on Test Set and accumulate fold ensemble probabilities
    model.eval()
    fold_test_probs = []
    with torch.no_grad():
        for test_batch in test_loader:
            test_batch = test_batch.to(device)
            out = model(test_batch)
            probs = F.softmax(out, dim=1).cpu().numpy()
            fold_test_probs.append(probs)
    test_ensemble_probs += np.concatenate(fold_test_probs, axis=0) / CNN_NUM_FOLDS

    cv_histories.append(fold_history)
    fold_time = time.time() - fold_start
    print(f"Fold {fold + 1:<2} | {tr_loss:<10.4f} | {tr_acc*100:<9.2f}% | {best_val_loss:<10.4f} | {fold_acc*100:<9.2f}% | {fold_time:<5.1f}s")

total_time = time.time() - start_time
overall_oof_acc = (oof_preds == y_train).mean() * 100
print("-" * 60)
print(f"All Folds Complete in {total_time:.1f}s | Overall Out-Of-Fold Accuracy: {overall_oof_acc:.2f}%")


In [ ]:
# ==============================================================================
# Cell 7: Out-Of-Fold Evaluation, Metrics & Confusion Matrix Plot
# ==============================================================================
oof_pred_labels = [ID2LABEL[idx] for idx in oof_preds]

# 1. Compute comprehensive classification metrics using the library function
oof_metrics = iml.compute_classification_metrics(
    y_true=y_train_labels,
    y_pred=oof_pred_labels,
    y_prob=oof_probs,
    class_names=ACTIVITY_CLASSES,
)

print(f"Overall OOF Accuracy:        {oof_metrics['accuracy_pct']}%")
print(f"Macro-Averaged F1 Score:     {oof_metrics['macro_f1']:.4f}")
print(f"Weighted F1 Score:           {oof_metrics['weighted_f1']:.4f}")
print(f"Balanced Accuracy:           {oof_metrics['balanced_accuracy']:.4f}")
print(f"OOF Multi-Class Log-Loss:    {oof_metrics['log_loss']:.4f}")

# 2. Display Per-Class Precision, Recall, and F1 Breakdown Table
per_class_df = pd.DataFrame(oof_metrics["per_class"]).T
print("\nPer-Class OOF Classification Report:")
display(per_class_df)

# 3. Plot Publication-Grade Confusion Matrix Heatmap
fig, ax = iml.plot_confusion_matrix(
    y_true=y_train_labels,
    y_pred=oof_pred_labels,
    class_names=ACTIVITY_CLASSES,
    normalize=True,
    title=f"1D-CNN 5-Fold OOF Confusion Matrix (Accuracy: {oof_metrics['accuracy_pct']}%)",
    save_name="1d_cnn_oof_confusion_matrix.png",
)
plt.show()

# 4. Plot Training & Validation Loss/Accuracy Curves across Folds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for f_idx, hist in enumerate(cv_histories):
    axes[0].plot(hist["val_loss"], label=f"Fold {f_idx+1}")
    axes[1].plot(hist["val_acc"], label=f"Fold {f_idx+1}")
axes[0].set_title("Validation Loss per Epoch", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_title("Validation Accuracy per Epoch", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Cell 8: Instant Test Set Evaluation Against SOTA Benchmark & Submission File Creation
# ==============================================================================
# 1. Convert ensemble test probabilities to predicted class names
test_pred_indices = np.argmax(test_ensemble_probs, axis=1)
test_pred_labels = [ID2LABEL[idx] for idx in test_pred_indices]

# 2. Offline Benchmark Evaluation: Evaluate directly against the 100% Kaggle-scoring SOTA benchmark
print("Evaluating 1D-CNN Predictions against 100% Kaggle-Scoring Benchmark:")
bench_eval = iml.evaluate_against_benchmark(
    predictions=test_pred_labels,
    plot_cm=True,
    save_cm="1d_cnn_test_benchmark_confusion_matrix.png",
    verbose=True,
)

print(f"\n1D-CNN Test Benchmark Accuracy: {bench_eval['accuracy_pct']}% ({bench_eval['matches']}/403 correct)")
print(f"Total Disagreements: {bench_eval['mismatches']}")
if bench_eval["mismatches"] > 0:
    print("\nTop Confusing Activity Pairs on Test Set:")
    print(bench_eval["confusion_pairs"].head(5))
    print("\nSample Disagreements:")
    display(bench_eval["mismatches_df"].head(10))

# 3. Create Validated Kaggle Submission CSV
submission_path = iml.create_submission_file(
    test_df=test_df,
    predictions=test_pred_labels,
    save_name="submission_1d_cnn.csv",
    verbose=True,
)
print(f"\nSubmission ready for upload at: {submission_path}")
